
# Microbiome Metadata Annotation

This notebook presents an process of annotating the metadata of 186,464 microbiome samples from 482 studies (projects). The primary focus is to annotate this metadata with specific tags related to 'extraction kit' and 'collection methods'. Such information is found within fields such as "samp_mat_process" for extraction kits and "samp_collect_device", "env_medium", "isolation_source" for collection methods.

The outline of this notebook is as follows:

1. Metadata Preprocessing
2. Calculation of Similarity Scores to Tag Sample Descriptions with the Most Similar Values
3. Conclusion and Comments for Next Steps

### Preprocessing

The `metadata_blob` comprises key-value pairs. We extracted the values, disregarding the keys, and constructed a bag of words from them. During preprocessing, we removed numbers, stop words, words with a length less than 2, and punctuation marks. The resulting preprocessed text serves as the sample/study description for subsequent analysis.

### Cosin Similarity

Next, we calculate the similarity score betwee the study/sample descriptions and the unique values extracted for the extraction kit (using samp_mat_process as the key). These unique values are listed below:

* qiaamp fast dna stool kit qiagen
* zymobiomics dna kit
* stools processing device biomerieux combination dneasy powerlyzer powersoil kit qiagen
* centrifuged supernatant removed frozen
* none
* stools processing device biomerieux combination zymobiomics dna mini kit zymoresearch
* stools processing device biomerieux combination qiaamp fast dna stool kit qiagen
* zymobiomics dna mini kit zymoresearch
* collected
* dna extraction
* applicable
* homogenisation stomacher
* mobio powersoil well dna kit min beadbeating retsch
* zymo research quick dna fecal soil microbe microprep kit cat
* illumina miseq amplicon sequencing
* mo bio power soil kit
* processing applied
* genomic dna isolated using qiagen stool kit followed illumina miseq amplicon sequencing
* dneasy powerlyzer powersoil kit qiagen
* dna isolation amplification miseq bp sequencing
* zymobiomics dna microprep kit cat
* processed
* snap frozen
* stools processing device biomerieux combination nucleospin soil kit macherey nagel
* nucleospin soil kit macherey nagel
* missing
* micropipet
* unprocessed

Some of these values can be merged to generate a biologically relevant set for extraction kit.

To compute the similarity score between a sample description and each extraction kit term, we begin by calculating the cosine similarity between each word in the description and each word in an extraction kit term (which may consist of one or multiple words). Subsequently, we compute a weighted average of these similarity values, followed by z-transformation across both columns and rows of the resulting similarity matrix. The final outcome is presented as a single score that quantifies the similarity between each sample description and each term. Next, we find the maximum similarity score across terms and tag the sample by the corresponding term given that the score is higher than a estimated threshold. The table below presents the results of this computation. The first column lists the maximum similarity score across all extraction kit terms, along with its corresponding term in the second column. These results are displayed for study-level data to facilitate reviewing more diverse descriptions. The last column represents the preprocessed metadata, where fields have been removed.

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display


In [ ]:
base_dir = '../data/results/study_level/extraction_kit/'
df_samp = pd.read_csv(os.path.join(base_dir, 'labeled_metadata.csv'))
# sort the similarity score and plot the top 20 for comparison
df_samp = df_samp.sort_values(by='Max_Score', ascending=False)
req_cols = ['Sample_ID','metadata_blob', 'Corresponding_Term', 'Max_Score' ]
df_samp = df_samp[req_cols]
pd.set_option('display.max_colwidth', None)
display(df_samp.iloc[20:40])

In [ ]:
display(df_samp.iloc[460:480])

### Next Step (March 25)

After reviewing a considerable number of descriptions, we noticed that only a limited number of samples contain any information regarding the extraction kit. Regardless of the tagging method useed, it's not feasible to verify the tag using only the metadata.

One potential approach to assess this outcome is to label the dataset at the study level. This way, we can verify the tags for the positive studies.

## Updates (April 10)

In the next step, we tried to augment the incomplete descriptions in the metadata by leveraging pertinent information from the Material and Methods section of the respective publication. To achieve this, we extracted the associated PMID(s) for each microbiome project, utilizing a spreadsheet provided by the PIs team. Out of the 482 projects, PMIDs were listed for 187 projects. Next, we used PubTator to retrieve the full text of these manuscripts, successfully acquiring 184 out of 187 projects in XML format. These XML files were then parsed to identify terms related to the DNA extraction kit. These descriptions were then utilized to extract similarity scores, as previously described. The table below presents the top 20 scores. The first and second columns list the project ID and the extracted description from the manuscript, respectively. The third column contains the term recommended by the computed Max_Score as the tag for the extraction kit, while the fourth column lists terms related to the extraction kit that appeared in the project description. Essentially, if columns 3 and 4 agree with each other, it indicates that our scoring system accurately tagged the project with the extraction kit. The final column displays the computed Max_Score.

In [ ]:
base_dir = '../data/material_and_methods/study_level/extraction_kit'
df_samp = pd.read_csv(os.path.join(base_dir, 'labeled_metadata.csv'))
# sort the similarity score and plot the top 20 for comparison
df_samp = df_samp.sort_values(by='Max_Score', ascending=False)
req_cols = ['project_ID','Description', 'Corresponding_Term', 'Matching Terms', 'Max_Score' ]
df_samp = df_samp[req_cols]
pd.set_option('display.max_colwidth', None)
display(df_samp.iloc[1:20])

In [ ]:
# distribution of similarity scores
plt.figure(figsize=(4, 4)) 
plt.boxplot(df_samp['Max_Score'])
plt.ylabel('similarity_score')
plt.title('study level')
plt.show()